In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from scipy.stats import pearsonr
from statsmodels.tsa.seasonal import seasonal_decompose
import os

warnings.filterwarnings("ignore")

# Gap-breaking utilities

def drop_isolated_points(dates, values, max_gap_days=45):
    dates  = pd.to_datetime(dates)
    values = np.asarray(values, dtype=float)
    days   = dates.to_series().reset_index(drop=True)
    keep   = np.ones(len(days), dtype=bool)
    for i in range(len(days)):
        prev_gap = (days[i] - days[i - 1]).days if i > 0             else np.inf
        next_gap = (days[i + 1] - days[i]).days if i < len(days) - 1 else np.inf
        if prev_gap > max_gap_days and next_gap > max_gap_days:
            keep[i] = False
    return dates[keep], values[keep]


def break_at_gaps(dates, values, max_gap_days=45):
    dates   = pd.to_datetime(dates)
    values  = np.asarray(values, dtype=float)
    dt_days = dates.to_series().diff().dt.total_seconds().div(86400).to_numpy()
    gap_idx = np.where(dt_days > max_gap_days)[0]
    if len(gap_idx) == 0:
        return dates.to_numpy(), values
    dates_out = dates.to_numpy().astype("datetime64[ns]")
    vals_out  = values.copy()
    for i, idx in enumerate(gap_idx):
        insert_at = idx + i
        dates_out = np.insert(dates_out, insert_at, dates_out[insert_at - 1])
        vals_out  = np.insert(vals_out,  insert_at, np.nan)
    return dates_out, vals_out


# Seasonal decomposition

def remove_seasonal_component(tec_data, period=12):
    """Remove the seasonal component from monthly TEC data via additive decomposition."""
    tec_data_seasonal = tec_data.copy()
    tec_data_seasonal = tec_data_seasonal.sort_values('YearMonth')
    tec_data_seasonal.set_index('YearMonth', inplace=True)

    original_series = tec_data_seasonal['VTEC (TECU)']
    total_points   = len(original_series)
    missing_points = original_series.isna().sum()
    missing_pct    = (missing_points / total_points) * 100

    tec_series = original_series.interpolate(method='cubic')

    print(f"  Interpolation statistics:")
    print(f"    Total monthly data points : {total_points}")
    print(f"    Missing data points       : {missing_points}")
    print(f"    Percentage interpolated   : {missing_pct:.2f}%")

    if len(tec_series) < period * 2:
        print("Warning: Not enough data for seasonal decomposition.")
        return tec_data_seasonal.reset_index()

    decomposition = seasonal_decompose(tec_series, model='additive',
                                       period=period, extrapolate_trend='freq')

    tec_data_seasonal['seasonal_component']     = decomposition.seasonal
    tec_data_seasonal['trend_component_decomp'] = decomposition.trend
    tec_data_seasonal['residual_component']     = decomposition.resid
    tec_data_seasonal['VTEC (TECU)']            = (tec_data_seasonal['VTEC (TECU)']
                                                    - tec_data_seasonal['seasonal_component'])
    return tec_data_seasonal.reset_index()


# Combined 2×2 residual analysis plot


def plot_residual_analysis(final_data):
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

    def _plot_series(dates, values):
        """Apply gap-breaking for monthly; plot directly for yearly (no gaps)."""
        dates_np = pd.to_datetime(np.asarray(dates)).to_numpy()
        vals_np  = np.asarray(values, dtype=float)
        n = len(dates_np)
        if n < 30:   # yearly — skip gap helpers
            return dates_np, vals_np
        dc, vc = drop_isolated_points(dates_np, vals_np)
        return break_at_gaps(np.asarray(dc), np.asarray(vc, dtype=float))

    # (1) Residuals vs Time
    d_r, y_r = _plot_series(final_data['YearMonth'].values, final_data['residuals'].values)
    ax1.plot(d_r, y_r, 'o-', alpha=0.7)
    ax1.axhline(y=0, color='r', linestyle='--', linewidth=1)
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Residuals (TECU)')
    ax1.set_title('Residuals vs Time')
    ax1.grid(True, alpha=0.3)

    # (2) Observed vs Predicted
    ax2.scatter(final_data['model_prediction'].values, final_data['VTEC (TECU)'].values, alpha=0.7)
    min_val = min(final_data['model_prediction'].min(), final_data['VTEC (TECU)'].min())
    max_val = max(final_data['model_prediction'].max(), final_data['VTEC (TECU)'].max())
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    corr_r, _ = pearsonr(final_data['model_prediction'].values, final_data['VTEC (TECU)'].values)
    ax2.text(0.05, 0.95, f'r = {corr_r:.3f}',
             transform=ax2.transAxes, fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax2.set_xlabel('Predicted TEC (TECU)')
    ax2.set_ylabel('Observed TEC (TECU)')
    ax2.set_title('Observed vs Predicted TEC')
    ax2.grid(True, alpha=0.3)

    # (3) Solar-corrected TEC + long-term trend
    d_sc, y_sc = _plot_series(final_data['YearMonth'].values,
                              final_data['solar_corrected_tec'].values)
    ax3.plot(d_sc, y_sc, 'o-', color='red', alpha=0.7, label='Solar-corrected TEC')
    ax3.plot(final_data['YearMonth'].values, final_data['trend_component'].values,
             '--', color='green', alpha=0.7, label='Long-term trend')
    ax3.set_xlabel('Year')
    ax3.set_ylabel('TEC (TECU)')
    ax3.set_title('Solar-Corrected TEC Trend')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # (4) Residual distribution
    ax4.hist(final_data['residuals'].values, bins=25, alpha=0.7, edgecolor='black')
    ax4.axvline(final_data['residuals'].values.mean(), color='r', linestyle='--',
                label=f'Mean: {final_data["residuals"].values.mean():.2f}')
    ax4.set_xlabel('Residuals (TECU)')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Distribution of Residuals')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


# Partial R² utility

def compute_partial_r2(results_full, merged_data, proxy_cols):
    """
    Compute partial R² for each proxy by comparing the full model R² against
    a reduced model that drops that proxy.  Also returns semi-partial R².
    """
    r2_full  = results_full.rsquared
    partial  = {}

    for col in proxy_cols:
        reduced_cols = [c for c in proxy_cols if c != col] + ['YearMonth_centered', 'const']
        X_red = merged_data[['YearMonth_centered'] + [c for c in proxy_cols if c != col]]
        X_red = sm.add_constant(X_red)
        r2_red = sm.OLS(merged_data['VTEC (TECU)'], X_red).fit().rsquared
        # partial R²  = (R²_full − R²_reduced) / (1 − R²_reduced)
        partial_r2      = (r2_full - r2_red) / (1 - r2_red) if (1 - r2_red) > 0 else np.nan
        # semi-partial R² = R²_full − R²_reduced  (unique variance contribution)
        semi_partial_r2 = r2_full - r2_red
        partial[col]    = {'partial_r2': partial_r2, 'semi_partial_r2': semi_partial_r2}

    return partial

# Main analysis function

def run_multiproxy_analysis(tec_file_path, f107_file_path, mgii_file_path,
                            sunspot_file_path, begin_date='2000-01-01',
                            end_date='2024-12-31', resample_freq='M',
                            station_name='SUTH'):
    """
    Multi-proxy regression:
        TEC = const + β₁·t + β₂·F10.7 + β₃·MgII + β₄·SNN + ε

    resample_freq : 'M' for monthly, 'Y' for yearly
    station_name  : short station identifier used in plot titles and filenames
    """

    label        = 'Monthly' if resample_freq == 'M' else 'Yearly'
    station_label = station_name.upper()

    def normalise_yearly(df):
        """Snap all yearly timestamps to Jan 01 of their year so merge_asof aligns correctly."""
        if resample_freq == 'Y':
            df['YearMonth'] = pd.to_datetime(df['YearMonth'].dt.year.astype(str) + '-01-01')
        return df

    def load_tec(file_path, begin, end):
        tec = pd.read_csv(file_path)
        tec = tec.rename(columns={'Time (UTC)': 'YearMonth'})
        tec.replace(999.99, np.nan, inplace=True)
        tec = tec.dropna()
        tec = tec[(tec['YearMonth'] >= begin) & (tec['YearMonth'] <= end)]
        tec['YearMonth'] = pd.to_datetime(tec['YearMonth'])
        tec = tec[
            (tec['YearMonth'].dt.time >= pd.to_datetime('10:00:00').time()) &
            (tec['YearMonth'].dt.time <= pd.to_datetime('14:00:00').time())
        ]
        min_val = tec['VTEC (TECU)'].min()
        tec = tec[(tec['VTEC (TECU)'] >= min_val) & (tec['VTEC (TECU)'] <= 100)]
        tec.set_index('YearMonth', inplace=True)

        # Always build monthly means first (needed for seasonal removal)
        tec_monthly = tec.resample('M').mean().reset_index()
        print("  Removing seasonal component...")
        tec_monthly = remove_seasonal_component(tec_monthly, period=12)

        if resample_freq == 'Y':
            tec_monthly.set_index('YearMonth', inplace=True)
            tec_out = tec_monthly[['VTEC (TECU)']].resample('Y').mean().reset_index()
        else:
            tec_out = tec_monthly[['YearMonth', 'VTEC (TECU)',
                                    'seasonal_component',
                                    'trend_component_decomp',
                                    'residual_component']]
        return normalise_yearly(tec_out)

    def load_f107(file_path, begin, end):
        f107 = pd.read_csv(file_path, header=None,
                           names=['fractional year', 'month', 'day',
                                  'Solar Flux Units (SFU)', 'index'],
                           delimiter=r'\s+')
        f107['fractional year'] = pd.to_numeric(f107['fractional year'], errors='coerce')
        f107['year']     = f107['fractional year'].astype(int)
        f107['YearMonth'] = pd.to_datetime(f107[['year', 'month', 'day']])
        f107 = f107[(f107['YearMonth'] >= begin) & (f107['YearMonth'] <= end)]
        period_key = f107['YearMonth'].dt.to_period(resample_freq)
        f107_out = (f107.groupby(period_key)['Solar Flux Units (SFU)']
                        .mean().reset_index())
        f107_out['YearMonth'] = f107_out['YearMonth'].dt.to_timestamp()
        return normalise_yearly(f107_out)

    def load_mgii(file_path, begin, end):
        mgii = pd.read_csv(file_path, delim_whitespace=True,
                           header=None, comment=';',
                           names=['fractional_year', 'month', 'day',
                                  'mgii_index', 'uncertainty', 'source_flag'])
        mgii['mgii_index']      = pd.to_numeric(mgii['mgii_index'],      errors='coerce')
        mgii['fractional_year'] = pd.to_numeric(mgii['fractional_year'], errors='coerce')
        mgii = mgii.dropna(subset=['mgii_index', 'fractional_year'])
        mgii['Year']     = mgii['fractional_year'].astype(int)
        mgii['YearMonth'] = pd.to_datetime(mgii[['Year', 'month', 'day']])
        mgii = mgii[(mgii['YearMonth'] >= pd.to_datetime(begin)) &
                    (mgii['YearMonth'] <= pd.to_datetime(end))]
        mgii.set_index('YearMonth', inplace=True)
        mgii_out = mgii['mgii_index'].resample(resample_freq).mean().reset_index()
        return normalise_yearly(mgii_out)

    def load_sunspot(file_path, begin, end):
        sun = pd.read_csv(file_path, delim_whitespace=True, header=None,
                          names=['Year', 'Month', 'Day', 'FractionalYear', 'Mean-SNN',
                                 'DailySTD', 'Number of observations',
                                 'Definitive/provisional indicator'])
        sun['YearMonth'] = (sun['Year'].astype(str) + '-'
                            + sun['Month'].astype(str).str.zfill(2) + '-'
                            + sun['Day'].astype(str).str.zfill(2))
        sun = sun[(sun['YearMonth'] >= begin) & (sun['YearMonth'] <= end)]
        sun['YearMonth'] = pd.to_datetime(sun['YearMonth'])
        sun.set_index('YearMonth', inplace=True)
        sun_out = sun[['Mean-SNN']].resample(resample_freq).mean().dropna().reset_index()
        return normalise_yearly(sun_out)

    #Regression

    def perform_regression(tec, f107, mgii, sun):
        # Merge all four datasets.
        # Yearly: all timestamps are normalised to Jan 01 → exact merge is safe.
        # Monthly: timestamps may differ by a few days → use merge_asof.
        if resample_freq == 'Y':
            merged = f107.merge(tec,  on='YearMonth', how='inner')
            merged = merged.merge(mgii, on='YearMonth', how='inner')
            merged = merged.merge(sun,  on='YearMonth', how='inner')
        else:
            merged = pd.merge_asof(f107.sort_values('YearMonth'),
                                   tec.sort_values('YearMonth'),
                                   on='YearMonth')
            merged = pd.merge_asof(merged.sort_values('YearMonth'),
                                   mgii.sort_values('YearMonth'),
                                   on='YearMonth')
            merged = pd.merge_asof(merged.sort_values('YearMonth'),
                                   sun.sort_values('YearMonth'),
                                   on='YearMonth')
        merged = merged.dropna(subset=['VTEC (TECU)', 'Solar Flux Units (SFU)',
                                       'mgii_index', 'Mean-SNN'])

        if len(merged) < 5:
            raise ValueError("Insufficient overlapping data across all three proxies.")

        # Fractional year
        merged['YearMonth_frac'] = (merged['YearMonth'].dt.year
                                    + (merged['YearMonth'].dt.dayofyear - 1) / 365.25)

        # Centre time
        year_mean = merged['YearMonth_frac'].mean()
        merged['YearMonth_centered'] = merged['YearMonth_frac'] - year_mean

        # Standardise each proxy independently (z-score)
        for raw_col, scaled_col in [('Solar Flux Units (SFU)', 'f107_scaled'),
                                    ('mgii_index',             'mgii_scaled'),
                                    ('Mean-SNN',               'snn_scaled')]:
            col_mean = merged[raw_col].mean()
            col_std  = merged[raw_col].std()
            merged[scaled_col] = (merged[raw_col] - col_mean) / col_std
            merged[f'_{scaled_col}_std'] = col_std   # store for back-transform

        # Full multi-proxy OLS
        X = sm.add_constant(merged[['YearMonth_centered',
                                    'f107_scaled', 'mgii_scaled', 'snn_scaled']])
        results = sm.OLS(merged['VTEC (TECU)'], X).fit()

        const     = results.params['const']
        coef_year = results.params['YearMonth_centered']

        # Back-transform scaled coefficients to original proxy units
        coef_f107 = results.params['f107_scaled'] / merged['_f107_scaled_std'].iloc[0]
        coef_mgii = results.params['mgii_scaled']  / merged['_mgii_scaled_std'].iloc[0]
        coef_snn  = results.params['snn_scaled']   / merged['_snn_scaled_std'].iloc[0]

        # Solar-corrected TEC (remove all three proxy contributions)
        merged['solar_corrected_tec'] = (
            merged['VTEC (TECU)']
            - coef_f107 * merged['Solar Flux Units (SFU)']
            - coef_mgii * merged['mgii_index']
            - coef_snn  * merged['Mean-SNN']
        )
        merged['trend_component']  = const + coef_year * merged['YearMonth_centered']
        merged['model_prediction'] = results.predict(X)
        merged['residuals']        = results.resid

        return merged, results, coef_year, coef_f107, coef_mgii, coef_snn, const

    #Partial R²

    def report_partial_r2(results, merged):
        proxy_scaled_cols = ['f107_scaled', 'mgii_scaled', 'snn_scaled']
        partial = compute_partial_r2(results, merged, proxy_scaled_cols)

        print(f"\n{'─'*50}")
        print("Partial R² — individual proxy contributions")
        print(f"{'─'*50}")
        labels = {'f107_scaled': 'F10.7', 'mgii_scaled': 'MgII', 'snn_scaled': 'Sunspot'}
        for col, vals in partial.items():
            name = labels.get(col, col)
            print(f"  {name:<12} partial R² = {vals['partial_r2']:.4f}   "
                  f"semi-partial R² = {vals['semi_partial_r2']:.4f}")
        return partial

    # Main 2-panel plot

    def create_main_plot(merged, coef_year, const):
        dates      = merged['YearMonth'].values
        tec_values = merged['VTEC (TECU)'].values
        model      = merged['model_prediction'].values
        resid      = merged['residuals'].values

        if resample_freq == 'M':
            # Monthly: apply gap-breaking. Convert to numpy arrays first to
            # avoid pandas index misalignment after drop_isolated_points.
            dates_np = pd.to_datetime(dates).to_numpy()
            dc, tc = drop_isolated_points(dates_np, tec_values)
            dc, mc = drop_isolated_points(dates_np, model)
            dc = np.asarray(dc); tc = np.asarray(tc, dtype=float)
            mc = np.asarray(mc, dtype=float)
            d_obs, y_obs = break_at_gaps(dc, tc)
            d_mod, y_mod = break_at_gaps(dc, mc)
            dr, rr = drop_isolated_points(dates_np, resid)
            dr = np.asarray(dr); rr = np.asarray(rr, dtype=float)
            d_rb, y_rb = break_at_gaps(dr, rr)
        else:
            # Yearly: only ~25 points, no gaps — plot directly.
            d_obs, y_obs = dates, tec_values
            d_mod, y_mod = dates, model
            d_rb,  y_rb  = dates, resid

        fig, (ax1a, ax1b) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                          gridspec_kw={"height_ratios": [1.2, 1.0]})

        # Panel (a): Observed vs Model
        ax1a.plot(d_obs, y_obs, marker='o', linestyle='-', linewidth=1.2,
                  markersize=3.5, alpha=0.8, label='Observed TEC')
        ax1a.plot(d_mod, y_mod, marker='o', linestyle='-', linewidth=1.2,
                  markersize=3.0, alpha=0.7, label='Model TEC')

        corr_r, _ = pearsonr(tec_values, model)
        ax1a.text(0.02, 0.95, f'r = {corr_r:.3f}',
                  transform=ax1a.transAxes, fontsize=10, va='top',
                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax1a.set_ylabel('TEC (TECU)')
        ax1a.set_title(f'Multi-proxy model — {station_label} ({label}): '
                       f'(a) Observed vs Model, (b) Residuals & Long-term Trend')
        ax1a.legend(fontsize=10, ncol=2)
        ax1a.grid(True, alpha=0.3)

        # Panel (b): Residuals + long-term trend
        ax1b_twin = ax1b.twinx()

        ax1b.plot(d_rb, y_rb, color='steelblue', marker='o', linestyle='-',
                  linewidth=1.2, markersize=3.5, alpha=0.8, label='Residuals')
        ax1b.axhline(0, color='black', linewidth=1, alpha=0.5)
        ax1b.set_ylabel('Residuals (TECU)', color='black')
        ax1b.tick_params(axis='y', labelcolor='black')

        sign        = '+' if coef_year >= 0 else '-'
        trend_label = f'Long-term trend: TEC = {const:.2f} {sign} {abs(coef_year):.4f}·t'
        ax1b_twin.plot(merged['YearMonth'].values, merged['trend_component'].values,
                       color='green', linestyle='--', linewidth=2.2, alpha=0.9,
                       label=trend_label)
        ax1b_twin.set_ylabel('')
        ax1b_twin.tick_params(axis='y', labelcolor='none', length=0)

        ax1b.set_xlabel('Year')
        ax1b.grid(True, alpha=0.3)

        lines_b,  labels_b  = ax1b.get_legend_handles_labels()
        lines_bt, labels_bt = ax1b_twin.get_legend_handles_labels()
        ax1b.legend(lines_b + lines_bt, labels_b + labels_bt, fontsize=10)

        freq = 2 if resample_freq == 'M' else 4
        ax1b.xaxis.set_major_locator(mdates.YearLocator(base=freq))
        ax1b.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        plt.setp(ax1b.get_xticklabels(), rotation=45, ha='right')

        plt.tight_layout()
        return fig

    # Proxy contribution bar chart

    def create_contribution_plot(partial_r2, results):
        labels_map  = {'f107_scaled': 'F10.7', 'mgii_scaled': 'MgII', 'snn_scaled': 'Sunspot'}
        proxy_names = [labels_map[k] for k in partial_r2]
        semi_vals   = [partial_r2[k]['semi_partial_r2'] * 100 for k in partial_r2]
        part_vals   = [partial_r2[k]['partial_r2']      * 100 for k in partial_r2]

        colors = ['#2196F3', '#FF9800', '#4CAF50']

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Semi-partial R²
        bars = ax1.bar(proxy_names, semi_vals, color=colors, alpha=0.8, edgecolor='black')
        for bar, val in zip(bars, semi_vals):
            ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                     f'{val:.2f}%', ha='center', va='bottom', fontsize=11)
        ax1.set_ylabel('Semi-partial R² (%)')
        ax1.set_title('Unique variance explained per proxy\n(semi-partial R²)')
        ax1.set_ylim(0, max(semi_vals) * 1.25)
        ax1.grid(True, alpha=0.3, axis='y')

        # Partial R²
        bars2 = ax2.bar(proxy_names, part_vals, color=colors, alpha=0.8, edgecolor='black')
        for bar, val in zip(bars2, part_vals):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                     f'{val:.2f}%', ha='center', va='bottom', fontsize=11)
        ax2.set_ylabel('Partial R² (%)')
        ax2.set_title('Variance explained after controlling\nfor other predictors (partial R²)')
        ax2.set_ylim(0, max(part_vals) * 1.25)
        ax2.grid(True, alpha=0.3, axis='y')

        # Annotate with full model R²
        fig.suptitle(f'Proxy contributions — {station_label} {label} multi-proxy model  '
                     f'(Full model R² = {results.rsquared:.3f})',
                     fontsize=13, y=1.01)
        plt.tight_layout()
        return fig

    # Execution
    try:
        print(f"\nProcessing TEC data ({label})...")
        tec_data = load_tec(tec_file_path, begin_date, end_date)

        print(f"Processing F10.7 data ({label})...")
        f107_data = load_f107(f107_file_path, begin_date, end_date)

        print(f"Processing MgII data ({label})...")
        mgii_data = load_mgii(mgii_file_path, begin_date, end_date)

        print(f"Processing sunspot data ({label})...")
        sun_data = load_sunspot(sunspot_file_path, begin_date, end_date)

        print("Performing multi-proxy regression...")
        (merged, results, coef_year,
         coef_f107, coef_mgii, coef_snn, const) = perform_regression(
            tec_data, f107_data, mgii_data, sun_data)

        # ── Print summary ──
        corr_obs_model, _ = pearsonr(merged['VTEC (TECU)'], merged['model_prediction'])

        print(f"\n{'='*55}")
        print(f"MULTI-PROXY REGRESSION RESULTS — {station_label} ({label.upper()}, DESEASONALIZED)")
        print(f"{'='*55}")
        print(f"Intercept (const)        : {const:.4f} TECU")
        print(f"Trend coefficient        : {coef_year:.6f} TECU/year")
        print(f"F10.7 coefficient        : {coef_f107:.6f} TECU per SFU")
        print(f"MgII coefficient         : {coef_mgii:.4f} TECU per MgII unit")
        print(f"Sunspot coefficient      : {coef_snn:.6f} TECU per SNN unit")
        print(f"Observed vs Model r      : {corr_obs_model:.3f}")
        print(f"R-squared                : {results.rsquared:.4f}")
        print(f"Adjusted R-squared       : {results.rsquared_adj:.4f}")
        print(f"AIC                      : {results.aic:.2f}")
        print(f"BIC                      : {results.bic:.2f}")
        print(f"Number of data points    : {len(merged)}")
        print(f"\nOLS summary:")
        print(results.summary())

        # ── Partial R² ──
        partial_r2 = report_partial_r2(results, merged)

        # ── Plots ──
        print("\nCreating plots...")
        main_fig         = create_main_plot(merged, coef_year, const)
        contribution_fig = create_contribution_plot(partial_r2, results)
        residual_fig     = plot_residual_analysis(merged)

        return {
            'merged_data'    : merged,
            'results'        : results,
            'coefficients'   : {
                'const'    : const,
                'year'     : coef_year,
                'f107'     : coef_f107,
                'mgii'     : coef_mgii,
                'snn'      : coef_snn,
            },
            'partial_r2'     : partial_r2,
            'plots'          : {
                'main'         : main_fig,
                'contribution' : contribution_fig,
                'residual'     : residual_fig,
            },
        }

    except Exception as e:
        print(f"Error in multi-proxy analysis ({label}): {e}")
        import traceback; traceback.print_exc()
        return None


# ─────────────────────────────────────────────────────────────
# Station configuration
# Add or remove stations here. Each entry is a dict with:
#   name        : short identifier used in filenames and plot titles
#   tec_file    : path to the station TEC data file
#   begin_date  : start of analysis period (YYYY-MM-DD)
#   end_date    : end   of analysis period (YYYY-MM-DD)
#
# The three solar proxy files (F10.7, MgII, Sunspot) are shared
# across all stations — update the paths once below.
# ─────────────────────────────────────────────────────────────

STATIONS = [
    {
        'name'       : 'SUTH',
        'tec_file'   : '/home/user/Documents/MScResearch/2024/Software/ionolabtecv1.41/30s_interval_data(suth).txt',
        'begin_date' : '2000-01-01',
        'end_date'   : '2024-12-31',
    },
    # ── Add more stations ──────────────────────────────
    # {
    #     'name'       : 'HRAO',
    #     'tec_file'   : '/path/to/30s_interval_data(hrao).txt',
    #     'begin_date' : '2000-01-01',
    #     'end_date'   : '2024-12-31',
    # },
    # {
    #     'name'       : 'HERM',
    #     'tec_file'   : '/path/to/30s_interval_data(herm).txt',
    #     'begin_date' : '2000-01-01',
    #     'end_date'   : '2024-12-31',
    # },
]

# ── Shared solar proxy files (same for all stations) ─────────
F107_FILE    = 'F10.7 dataII.txt'
MGII_FILE    = 'MgII data.txt'
SUNSPOT_FILE = 'R.txt'


def main():
    for station in STATIONS:
        name  = station['name']
        begin = station['begin_date']
        end   = station['end_date']

        print(f"\n{'#'*60}")
        print(f"  Station: {name}  ({begin} → {end})")
        print(f"{'#'*60}")

        for freq, freq_label in [('M', 'monthly'), ('Y', 'yearly')]:
            print(f"\n  ── {freq_label.capitalize()} resolution ──")

            results = run_multiproxy_analysis(
                tec_file_path     = station['tec_file'],
                f107_file_path    = F107_FILE,
                mgii_file_path    = MGII_FILE,
                sunspot_file_path = SUNSPOT_FILE,
                begin_date        = begin,
                end_date          = end,
                resample_freq     = freq,
                station_name      = name,
            )

            if results is not None:
                tag = f"{name.lower()}_{freq_label}"   # e.g. suth_monthly
                print(f"\nSaving {freq_label} outputs for {name}...")

                for plot_key, plot_suffix in [('main',         'main'),
                                               ('contribution', 'contribution'),
                                               ('residual',     'residual')]:
                    base = f'tec_multiproxy_{tag}_{plot_suffix}'
                    results['plots'][plot_key].savefig(f'{base}.png', dpi=300, bbox_inches='tight')
                    results['plots'][plot_key].savefig(f'{base}.pdf', bbox_inches='tight')
                    print(f"Saved: {base}.png/.pdf")
                    plt.close(results['plots'][plot_key])

                results['merged_data'].to_csv(
                    f'tec_multiproxy_{tag}_results.csv', index=False)
                print(f"Data saved: tec_multiproxy_{tag}_results.csv")

                print(f"\n{name} {freq_label} analysis completed successfully!")
            else:
                print(f"{name} {freq_label} analysis failed.")


if __name__ == "__main__":
    main()


  ── Monthly resolution ──

Processing TEC data (Monthly)...
  Removing seasonal component...
  Interpolation statistics:
    Total monthly data points : 288
    Missing data points       : 18
    Percentage interpolated   : 6.25%
Processing F10.7 data (Monthly)...
Processing MgII data (Monthly)...
Processing sunspot data (Monthly)...
Performing multi-proxy regression...

Intercept (const)        : 27.9840 TECU
Trend coefficient        : -0.476298 TECU/year
F10.7 coefficient        : -0.003354 TECU per SFU
MgII coefficient         : 2363.2949 TECU per MgII unit
Sunspot coefficient      : 0.030221 TECU per SNN unit
Observed vs Model r      : 0.954
R-squared                : 0.9107
Adjusted R-squared       : 0.9094
AIC                      : 1693.16
BIC                      : 1711.35
Number of data points    : 281

OLS summary:
                            OLS Regression Results                            
Dep. Variable:            VTEC (TECU)   R-squared:                       0.911
Mod